In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
paper_4.2.4.4_kfixed_perm_bootstrap_onlyMat.py
----------------------------------------------
目的：
- 「k=2だから高く見える」論点に対して、統計的に反証／補強するための検証パッケージ。
- OH（材料名） vs FP（フラグメント）整合のサンプル単位 ARI を、
  (i) k固定ストラタ内で index を比較、
  (ii) 置換検定で有意性（p値）評価、
  (iii) ブートストラップで95% CIを推定、
  (iv) クラスタサイズ不均衡も併記、
  する。

入力（既存）:
  ROOT/
    ├─ features_OH_onlyMat.csv
    ├─ features_FP_onlyMat.csv
    ├─ figs_OH/ClusterAssign_(top3|cumeig)_(index)_OH_*.csv
    └─ figs_FP/ClusterAssign_(top3|cumeig)_(index)_FP_*.csv

出力（章番号入り＆用途明記）:
  ROOT/paper_4.2.4.4_OHFP/validation/
    ├─ Table_4.2.4.4_validation_per-condition.csv
    ├─ Table_4.2.4.4_validation_k-fixed_summary.csv
    ├─ Fig_4.2.4.4_perm-hist_{mode}_{index}.(png|pdf)      # 置換ヒスト＋観測ARI
    ├─ Fig_4.2.4.4_k2_only_ARIs_by-index_{mode}.(png|pdf)  # kOH=kFP=2 限定
    └─ Fig_4.2.4.4_kNot2_ARIs_by-index_{mode}.(png|pdf)    # それ以外

注意：
- 計算コスト：Permutation/Bootstrap の反復回数は下の定数で調整可能。
"""

from pathlib import Path
import re, os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
from sklearn.metrics import adjusted_rand_score
from scipy.spatial.distance import cosine

# ========= 設定 =========
ROOT = Path("/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No")
FIGS_OH = ROOT / "figs_OH"
FIGS_FP = ROOT / "figs_FP"
FEATURES_OH = ROOT / "features_OH_onlyMat.csv"
FEATURES_FP = ROOT / "features_FP_onlyMat.csv"

OUT_ROOT  = ROOT / "paper_4.2.4.4_OHFP" / "validation"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

DIMS    = ["top3","cumeig"]
INDICES = ["silhouette","dunn","gap","ch","db","ptbiserial"]

# 反復回数（必要に応じて増減）
N_PERM = 500       # 置換検定
N_BOOT = 500       # ブートストラップCI
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

DPI = 300

# ========= フォント =========
def set_font():
    cand = ["Noto Sans CJK JP","Noto Sans JP","Source Han Sans JP","IPAexGothic","Yu Gothic","Meiryo"]
    have = set()
    for p in fm.findSystemFonts(fontext="ttf"):
        try: have.add(fm.FontProperties(fname=p).get_name())
        except: pass
    for w in cand:
        if any(w.lower() in h.lower() for h in have):
            matplotlib.rcParams["font.family"] = w
            break
    matplotlib.rcParams["axes.unicode_minus"] = False

# ========= IO =========
def read_csv_guess(path: Path) -> pd.DataFrame:
    for enc in ("utf-8","utf-8-sig","cp932"):
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            continue
    raise RuntimeError(f"Failed to read: {path}")

def load_features(path: Path) -> pd.DataFrame:
    df = read_csv_guess(path)
    if df.columns[0].lower() != "sample_id":
        df = df.rename(columns={df.columns[0]:"sample_id"})
    X = df.set_index("sample_id").apply(pd.to_numeric, errors="coerce").fillna(0.0)
    return (X != 0).astype(int)

FN_RE = re.compile(
    r"^ClusterAssign_(?P<mode>top3|cumeig)_(?P<index>silhouette|dunn|gap|ch|db|ptbiserial)_(?P<ds>OH|FP)_(?P<date>\d{8})_(?P<hm>\d{4})\.csv$"
)

def latest_by_mode_index(fig_dir: Path, ds: str):
    latest = {}
    for p in fig_dir.glob("ClusterAssign_*.csv"):
        m = FN_RE.match(p.name)
        if not m or m["ds"] != ds: 
            continue
        key = (m["mode"], m["index"])
        ts  = f"{m['date']}_{m['hm']}"
        if key not in latest or ts > latest[key][0]:
            latest[key] = (ts, p)
    return {k: v[1] for k,v in latest.items()}

def read_var_cluster(path: Path) -> pd.Series:
    df = read_csv_guess(path)
    cols = {c.lower(): c for c in df.columns}
    vcol = cols.get("variable", df.columns[0])
    ccol = cols.get("cluster",  df.columns[-1])
    s = pd.Series(df[ccol].values, index=df[vcol].astype(str).values, name="cluster")
    try: s = s.astype(int)
    except: pass
    return s

# ========= ラベル導出（サンプル → 優勢クラスタ） =========
def derive_sample_labels(features: pd.DataFrame, var2clu: pd.Series, min_votes=1) -> pd.Series:
    common = [v for v in features.columns if v in var2clu.index]
    if not common:
        return pd.Series(index=features.index, dtype="float64")
    clus = pd.Series(var2clu.loc[common]).astype(str)
    uniq = sorted(clus.unique())
    mat = features[common].values
    key2idx = {k:i for i,k in enumerate(uniq)}
    col2cidx = np.array([key2idx[c] for c in clus.values], dtype=int)
    votes = np.zeros((features.shape[0], len(uniq)), dtype=int)
    rows, cols = np.where(mat==1)
    for r, c in zip(rows, cols):
        votes[r, col2cidx[c]] += 1
    maxv = votes.max(axis=1)
    labels = np.where(maxv>=min_votes, votes.argmax(axis=1), -1)
    out = pd.Series(labels, index=features.index).replace({-1: np.nan})
    # 元クラスタはstrキー → int化を試行
    lbl_vals = [uniq[int(i)] if pd.notna(i) else np.nan for i in out.values]
    try:
        return pd.Series(lbl_vals, index=features.index).astype(int)
    except:
        return pd.Series(lbl_vals, index=features.index)

# ========= 指標：サイズ不均衡 =========
def cluster_size_stats(labels: pd.Series):
    s = labels.dropna().astype(int).value_counts(normalize=True)
    if s.empty:
        return np.nan, np.nan
    max_ratio = float(s.max())
    # Gini
    p = s.values
    gini = 1.0 - np.sum(p**2)
    return max_ratio, float(gini)

# ========= 置換／ブート関数 =========
def permutation_pvalue(yA: np.ndarray, yB: np.ndarray, n_perm=N_PERM):
    obs = adjusted_rand_score(yA, yB)
    null = np.zeros(n_perm, dtype=float)
    for i in range(n_perm):
        perm = rng.permutation(yB)
        null[i] = adjusted_rand_score(yA, perm)
    p = float((np.sum(null >= obs) + 1) / (n_perm + 1))  # 右片側
    return obs, null, p

def bootstrap_ci(yA: np.ndarray, yB: np.ndarray, n_boot=N_BOOT, alpha=0.05):
    n = len(yA)
    vals = np.zeros(n_boot, dtype=float)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        vals[i] = adjusted_rand_score(yA[idx], yB[idx])
    lo = float(np.quantile(vals, alpha/2))
    hi = float(np.quantile(vals, 1-alpha/2))
    return (lo, hi), vals

# ========= 図 =========
def plot_perm_hist(null_vals: np.ndarray, obs_ari: float, mode: str, index: str, outdir: Path):
    set_font()
    fig, ax = plt.subplots(figsize=(7,4), dpi=DPI)
    ax.hist(null_vals, bins=30, alpha=0.9)
    ax.axvline(obs_ari, color="red", linestyle="--", linewidth=1.5, label=f"Observed ARI={obs_ari:.3f}")
    ax.set_title(f"Fig 4.2.4.4: Permutation null vs observed — {mode} × {index}")
    ax.set_xlabel("ARI under label permutation (null)"); ax.set_ylabel("Count")
    ax.legend()
    fig.tight_layout()
    fig.savefig(outdir / f"Fig_4.2.4.4_perm-hist_{mode}_{index}.png", bbox_inches="tight")
    fig.savefig(outdir / f"Fig_4.2.4.4_perm-hist_{mode}_{index}.pdf", bbox_inches="tight")
    plt.close(fig)

def plot_kfixed_bars(df: pd.DataFrame, mode: str, k_filter: str, outdir: Path):
    """
    k_filter: "k2_only" or "kNot2"
    df: rows of a single mode with columns [index, ARI, p_perm]
    """
    if df.empty: return
    set_font()
    df = df.sort_values("index")
    fig, ax = plt.subplots(figsize=(9,4), dpi=DPI)
    x = np.arange(len(df))
    colors = ["#4E79A7" if p<=0.05 else "#9FA8B3" for p in df["p_perm"]]
    ax.bar(x, df["ARI"], color=colors, alpha=0.95)
    for xi, r in enumerate(df.itertuples()):
        ax.text(xi, r.ARI+0.01, f"{r.ARI:.2f}\np={r.p_perm:.3f}", ha="center", va="bottom", fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(df["index"], rotation=35, ha="right")
    ax.set_ylim(0,1.05); ax.set_ylabel("ARI (OH dominant vs FP dominant)")
    ax.set_title(f"Fig 4.2.4.4: ARI by index — {mode} ({'kOH=kFP=2' if k_filter=='k2_only' else 'k≠2含む'})")
    ax.grid(axis="y", linestyle="--", linewidth=0.4, alpha=0.6)
    fname_core = f"Fig_4.2.4.4_{k_filter}_ARIs_by-index_{mode}"
    fig.tight_layout()
    fig.savefig(outdir / f"{fname_core}.png", bbox_inches="tight")
    fig.savefig(outdir / f"{fname_core}.pdf", bbox_inches="tight")
    plt.close(fig)

# ========= メイン =========
def main():
    set_font()

    # features（共通サンプルにアライン）
    Xoh0 = load_features(FEATURES_OH)
    Xfp0 = load_features(FEATURES_FP)
    common_samples = Xoh0.index.intersection(Xfp0.index)
    if len(common_samples) == 0:
        raise RuntimeError("No common samples between features_OH_onlyMat and features_FP_onlyMat.")
    Xoh0 = Xoh0.loc[common_samples]
    Xfp0 = Xfp0.loc[common_samples]

    # 既存クラスタ（最新）を (mode,index) で対応付け
    oh_latest = latest_by_mode_index(FIGS_OH, "OH")
    fp_latest = latest_by_mode_index(FIGS_FP, "FP")
    keys = [(m,i) for m in DIMS for i in INDICES if (m,i) in oh_latest and (m,i) in fp_latest]
    if not keys:
        raise RuntimeError("No common (mode,index) between figs_OH and figs_FP.")

    # 出力収集
    rows_cond = []

    # 条件ごと解析
    for (mode, index) in keys:
        oh_assign = read_var_cluster(oh_latest[(mode,index)])
        fp_assign = read_var_cluster(fp_latest[(mode,index)])

        # サンプル優勢クラスタ
        Xoh = Xoh0[[c for c in Xoh0.columns if c in oh_assign.index]]
        Xfp = Xfp0[[c for c in Xfp0.columns if c in fp_assign.index]]
        lab_oh = derive_sample_labels(Xoh, oh_assign, min_votes=1)
        lab_fp = derive_sample_labels(Xfp, fp_assign, min_votes=1)

        mask = lab_oh.notna() & lab_fp.notna()
        if mask.sum() < 5:
            # サンプルが少なすぎる場合はスキップ
            continue
        yA = lab_oh[mask].astype(int).values
        yB = lab_fp[mask].astype(int).values

        # 観測値
        ari_obs = float(adjusted_rand_score(yA, yB))
        k_oh = int(pd.Series(oh_assign.dropna().astype(str).astype(int)).nunique())
        k_fp = int(pd.Series(fp_assign.dropna().astype(str).astype(int)).nunique())
        n_eff = int(mask.sum())

        # サイズ不均衡
        maxr_oh, gini_oh = cluster_size_stats(pd.Series(yA))
        maxr_fp, gini_fp = cluster_size_stats(pd.Series(yB))

        # 置換検定
        ari_obs2, null_vals, p_perm = permutation_pvalue(yA, yB, n_perm=N_PERM)
        # ブートCI
        (ci_lo, ci_hi), boot_vals = bootstrap_ci(yA, yB, n_boot=N_BOOT, alpha=0.05)

        # 保存行
        rows_cond.append({
            "mode": mode, "index": index,
            "n_samples": n_eff, "k_OH": k_oh, "k_FP": k_fp,
            "ARI": ari_obs, "p_perm": p_perm,
            "boot_CI_lo": ci_lo, "boot_CI_hi": ci_hi,
            "OH_maxClusterRatio": maxr_oh, "OH_gini": gini_oh,
            "FP_maxClusterRatio": maxr_fp, "FP_gini": gini_fp
        })

        # 図：置換ヒスト
        plot_perm_hist(null_vals, ari_obs, mode, index, OUT_ROOT)

    # 条件別テーブル出力
    df_cond = pd.DataFrame(rows_cond).sort_values(["mode","index"])
    out_csv_cond = OUT_ROOT / "Table_4.2.4.4_validation_per-condition.csv"
    df_cond.to_csv(out_csv_cond, index=False, encoding="utf-8-sig")
    print(f"[SAVE] {out_csv_cond}")

    # ====== k固定ストラタ集約 ======
    # kOH=kFP=2 と それ以外 に分けて、index ごとの平均ARI・有意率(p≤0.05)など
    def summarize_block(df):
        if df.empty: 
            return pd.DataFrame(columns=["mode","index","n_conditions","mean_ARI","median_ARI","sig_rate"])
        g = (df.groupby(["mode","index"], as_index=False)
               .agg(n_conditions=("ARI","size"),
                    mean_ARI=("ARI","mean"),
                    median_ARI=("ARI","median"),
                    sig_rate=("p_perm", lambda s: float((s<=0.05).mean()))))
        return g

    df_k2   = df_cond[(df_cond["k_OH"]==2) & (df_cond["k_FP"]==2)].copy()
    df_kNot2 = df_cond[~((df_cond["k_OH"]==2) & (df_cond["k_FP"]==2))].copy()

    sum_k2    = summarize_block(df_k2)
    sum_kNot2 = summarize_block(df_kNot2)
    sum_k2["k_stratum"]    = "kOH=kFP=2"
    sum_kNot2["k_stratum"] = "kOH/kFP≠2含む"
    df_kfix = pd.concat([sum_k2, sum_kNot2], ignore_index=True)
    out_csv_kfix = OUT_ROOT / "Table_4.2.4.4_validation_k-fixed_summary.csv"
    df_kfix.to_csv(out_csv_kfix, index=False, encoding="utf-8-sig")
    print(f"[SAVE] {out_csv_kfix}")

    # 図：k=2 限定／それ以外 の index 別 ARI バー（p≤0.05 を色分け）
    for mode in DIMS:
        plot_kfixed_bars(df_k2[df_k2["mode"]==mode][["index","ARI","p_perm"]], mode, "k2_only", OUT_ROOT)
        plot_kfixed_bars(df_kNot2[df_kNot2["mode"]==mode][["index","ARI","p_perm"]], mode, "kNot2", OUT_ROOT)

    print("\n✅ Done: validation outputs →", OUT_ROOT)

if __name__ == "__main__":
    main()

[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP/validation/Table_4.2.4.4_validation_per-condition.csv
[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP/validation/Table_4.2.4.4_validation_k-fixed_summary.csv


/tmp/ipykernel_3910997/1832358885.py:209: UserWarning: Glyph 21547 (\N{CJK UNIFIED IDEOGRAPH-542B}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/tmp/ipykernel_3910997/1832358885.py:209: UserWarning: Glyph 12416 (\N{HIRAGANA LETTER MU}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/tmp/ipykernel_3910997/1832358885.py:210: UserWarning: Glyph 21547 (\N{CJK UNIFIED IDEOGRAPH-542B}) missing from font(s) DejaVu Sans.
  fig.savefig(outdir / f"{fname_core}.png", bbox_inches="tight")
/tmp/ipykernel_3910997/1832358885.py:210: UserWarning: Glyph 12416 (\N{HIRAGANA LETTER MU}) missing from font(s) DejaVu Sans.
  fig.savefig(outdir / f"{fname_core}.png", bbox_inches="tight")
/tmp/ipykernel_3910997/1832358885.py:211: UserWarning: Glyph 21547 (\N{CJK UNIFIED IDEOGRAPH-542B}) missing from font(s) DejaVu Sans.
  fig.savefig(outdir / f"{fname_core}.pdf", bbox_inches="tight")
/tmp/ipykernel_3910997/1832358885.py:211: UserWarning: Glyph 12416 (\N{HIRAGANA LETTER MU}) missing from font


✅ Done: validation outputs → /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP/validation


/tmp/ipykernel_3910997/1832358885.py:211: UserWarning: Glyph 21547 (\N{CJK UNIFIED IDEOGRAPH-542B}) missing from font(s) DejaVu Sans.
  fig.savefig(outdir / f"{fname_core}.pdf", bbox_inches="tight")
/tmp/ipykernel_3910997/1832358885.py:211: UserWarning: Glyph 12416 (\N{HIRAGANA LETTER MU}) missing from font(s) DejaVu Sans.
  fig.savefig(outdir / f"{fname_core}.pdf", bbox_inches="tight")
